In [ ]:
# ============================================================
# PIPELINE CONSOLIDADO — Proyecto Inversiones Vs. Inflación
# ============================================================
!pip install yfinance pdfplumber --quiet

import traceback
import requests
import pandas as pd
import yfinance as yf
import re
import time
from datetime import date, timedelta

from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

sheet = gc.open('Procesamiento')
ws_consolidado = sheet.worksheet('Rendimiento_Consolidado')
ws_log = sheet.worksheet('_pipeline_log')
ws_errores = sheet.worksheet('_detalle_errores')

BASE_ARGENTINADATOS = 'https://api.argentinadatos.com/v1'
HEADERS_CAFCI = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
periodos = {'1 mes': 30, '3 meses': 90, '6 meses': 180, '12 meses': 365}
fecha_hora_corrida = pd.Timestamp.now().isoformat()

# ---------- Manejo de errores (Verónica + Facundo) ----------
detalle_errores_local = []

def fetch_con_reintentos(fetch_func, max_intentos=3, **kwargs):
    ultimo_error = None
    for intento in range(max_intentos):
        try:
            return fetch_func(**kwargs)
        except Exception as e:
            ultimo_error = str(e)
            if intento < max_intentos - 1:
                time.sleep(2 * (intento + 1))
    raise Exception(f'Fallo tras {max_intentos} intentos: {ultimo_error}')

def registrar_error(funcion, item, tipo_error, detalle):
    detalle_errores_local.append({'fecha_hora': fecha_hora_corrida, 'funcion': funcion, 'item': item, 'tipo_error': tipo_error, 'detalle': detalle})

def escribir_detalle_errores():
    if len(detalle_errores_local) == 0:
        print('Sin errores en esta corrida.')
        return
    df_errores = pd.DataFrame(detalle_errores_local)
    ws_errores.append_rows(df_errores.values.tolist())
    print(f'{len(df_errores)} errores registrados en _detalle_errores.')

# ---------- Categoría 1: Inflación ----------
def obtener_inflacion_mensual():
    r = requests.get(f'{BASE_ARGENTINADATOS}/finanzas/indices/inflacion', timeout=15)
    df = pd.DataFrame(r.json())
    df['fecha'] = pd.to_datetime(df['fecha'])
    return df.sort_values('fecha', ascending=False).reset_index(drop=True)

def inflacion_acumulada_ultimos_n_meses(df_inflacion, n_meses):
    ultimos = df_inflacion.head(n_meses)
    factor = 1.0
    for valor in ultimos['valor']:
        factor *= (1 + valor / 100)
    return factor - 1

periodo_a_meses = {'1 mes': 1, '3 meses': 3, '6 meses': 6, '12 meses': 12}

# ---------- Categoría 2: Plazo fijo ----------
def obtener_plazo_fijo_30_dias():
    r = requests.get(f'{BASE_ARGENTINADATOS}/finanzas/tasas/depositos30Dias', timeout=15)
    df = pd.DataFrame(r.json())
    df['fecha'] = pd.to_datetime(df['fecha'])
    return df.sort_values('fecha', ascending=False).reset_index(drop=True)

def rendimiento_plazo_fijo(df_pf, dias):
    fecha_corte = df_pf['fecha'].max() - pd.Timedelta(days=dias)
    ventana = df_pf[df_pf['fecha'] >= fecha_corte]
    if ventana.empty:
        return None
    tna_promedio = ventana['valor'].mean() / 100
    return tna_promedio * (dias / 365)

# ---------- Categoría 3: Money market / FCI (CAFCI) ----------
def parsear_rendimientos_cafci(texto_markdown, nombre_fondo, categoria):
    patron = r'\|\s*(7 días|1 mes|90 días|180 días|En el año|12 meses)\s*\|\s*([\d.,]+)\s*%\s*\|'
    matches = re.findall(patron, texto_markdown)
    dias_por_periodo = {'1 mes': 30, '90 días': 90, '180 días': 180, '12 meses': 365}
    nombre_periodo = {'1 mes': '1 mes', '90 días': '3 meses', '180 días': '6 meses', '12 meses': '12 meses'}
    resultados = []
    for periodo, tna_str in matches:
        if periodo not in dias_por_periodo:
            continue
        tna = float(tna_str.replace('.', '').replace(',', '.')) / 100
        dias = dias_por_periodo[periodo]
        resultados.append({'categoria': categoria, 'simbolo': nombre_fondo, 'periodo': nombre_periodo[periodo], 'dias': dias, 'rendimiento_periodo': tna * (dias / 365)})
    return resultados

def obtener_rendimientos_fondos_cafci(fondos_elegidos, categoria):
    resultados = []
    for nombre, info in fondos_elegidos.items():
        try:
            url = f"https://defuddle.md/https://estadisticas.cafci.org.ar/fondos/{info['fondo_id']}?clase={info['clase_id']}"
            r = fetch_con_reintentos(requests.get, url=url, headers=HEADERS_CAFCI, timeout=15)
            resultados.extend(parsear_rendimientos_cafci(r.text, nombre, categoria))
        except Exception as e:
            registrar_error('obtener_rendimientos_fondos_cafci', nombre, 'fetch_fallido', str(e))
    return resultados

fondos_fci = {
    'Cocos Ahorro': {'fondo_id': 1469, 'clase_id': 4337},
    'Mercado Fondo': {'fondo_id': 798, 'clase_id': 1982},
    'Ualintec Ahorro Pesos': {'fondo_id': 1307, 'clase_id': 3713},
}

# ---------- Categorías 4 y 5: CEDEARs y acciones locales (yfinance .BA) ----------
def rendimiento_periodo_accion(simbolo, dias):
    data = yf.Ticker(simbolo).history(period=f'{dias+10}d')
    data = data.dropna(subset=['Close'])
    if data.empty or len(data) < 2:
        raise Exception(f'Datos insuficientes para {simbolo} en ventana de {dias} días')
    return (data['Close'].iloc[-1] / data['Close'].iloc[0]) - 1

def obtener_rendimientos_ba(simbolos, periodos_dias, categoria):
    resultados = []
    for simbolo in simbolos:
        for periodo, dias in periodos_dias.items():
            try:
                r = fetch_con_reintentos(rendimiento_periodo_accion, simbolo=f'{simbolo}.BA', dias=dias)
                resultados.append({'categoria': categoria, 'simbolo': simbolo, 'periodo': periodo, 'dias': dias, 'rendimiento_periodo': r})
            except Exception as e:
                registrar_error('obtener_rendimientos_ba', simbolo, 'fetch_fallido', str(e))
    return resultados

# ---------- Categoría 6: Acciones USA en USD (solo informativo) ----------
def obtener_rendimientos_acciones_usa(simbolos, periodos_dias, categoria):
    resultados = []
    for simbolo in simbolos:
        for periodo, dias in periodos_dias.items():
            try:
                r = fetch_con_reintentos(rendimiento_periodo_accion, simbolo=simbolo, dias=dias)
                resultados.append({'categoria': categoria, 'simbolo': simbolo, 'periodo': periodo, 'dias': dias, 'rendimiento_periodo_usd': r})
            except Exception as e:
                registrar_error('obtener_rendimientos_acciones_usa', simbolo, 'fetch_fallido', str(e))
    return resultados

# ---------- Dólar MEP (para convertir acciones USA a ARS) ----------
def obtener_dolar_mep():
    r = requests.get('https://api.argentinadatos.com/v1/cotizaciones/dolares/bolsa', timeout=15)
    df = pd.DataFrame(r.json())
    df['fecha'] = pd.to_datetime(df['fecha'])
    return df.sort_values('fecha', ascending=False).reset_index(drop=True)

def variacion_dolar_periodo(df_dolar, dias):
    fecha_objetivo = df_dolar['fecha'].max() - pd.Timedelta(days=dias)
    df_dolar_copy = df_dolar.copy()
    df_dolar_copy['diff'] = (df_dolar_copy['fecha'] - fecha_objetivo).abs()
    fila_inicio = df_dolar_copy.loc[df_dolar_copy['diff'].idxmin()]
    precio_hoy = df_dolar_copy.iloc[0]['venta']
    precio_inicio = fila_inicio['venta']
    return (precio_hoy / precio_inicio) - 1

# ---------- Categoría 8: Acciones USA convertidas a ARS (sí entra a la comparación) ----------
def obtener_rendimientos_acciones_usa_en_ars(simbolos, periodos_dias, df_dolar):
    resultados = []
    for simbolo in simbolos:
        for periodo, dias in periodos_dias.items():
            try:
                r_usd = fetch_con_reintentos(rendimiento_periodo_accion, simbolo=simbolo, dias=dias)
                var_dolar = variacion_dolar_periodo(df_dolar, dias)
                r_ars = (1 + r_usd) * (1 + var_dolar) - 1
                resultados.append({
                    'categoria': 'accion_usa', 'simbolo': simbolo, 'periodo': periodo, 'dias': dias,
                    'rendimiento_periodo': r_ars, 'instrumento_dolarizado': True
                })
            except Exception as e:
                registrar_error('obtener_rendimientos_acciones_usa_en_ars', simbolo, 'fetch_fallido', str(e))
    return resultados

# ---------- Categoría 7: ONs (IOL) ----------
def obtener_token_iol(username, password):
    r = requests.post('https://api.invertironline.com/token',
                       data={'username': 'USERNAME', 'password': 'PASSWORD', 'grant_type': 'PASSWORD'},
                       timeout=15)
    return r.json()['access_token']

def obtener_historico_iol(simbolo, fecha_desde, fecha_hasta, access_token, mercado='bCBA'):
    headers = {'Authorization': f'Bearer {access_token}', 'Accept': 'application/json'}
    url = f'https://api.invertironline.com/api/{mercado}/Titulos/{simbolo}/Cotizacion/seriehistorica/{fecha_desde}/{fecha_hasta}/sinAjustar'
    r = requests.get(url, headers=headers, timeout=15)
    if r.status_code == 200:
        return r.json()
    raise Exception(f'{r.status_code}: {r.text[:200]}')

def rendimiento_periodo_on(serie, dias, tolerancia_dias=15):
    df = pd.DataFrame(serie)
    df['fechaHora'] = pd.to_datetime(df['fechaHora'], format='ISO8601')
    df = df.sort_values('fechaHora')
    fecha_objetivo = df['fechaHora'].max() - pd.Timedelta(days=dias)
    df['diff'] = (df['fechaHora'] - fecha_objetivo).abs()
    fila_inicio = df.loc[df['diff'].idxmin()]
    if fila_inicio['diff'] > pd.Timedelta(days=tolerancia_dias):
        return None
    return (df.iloc[-1]['ultimoPrecio'] / fila_inicio['ultimoPrecio']) - 1

def obtener_rendimientos_ons(tickers, periodos_dias, access_token):
    resultados = []
    fecha_hasta = date.today().strftime('%Y-%m-%d')
    fecha_desde = (date.today() - timedelta(days=370)).strftime('%Y-%m-%d')
    for ticker in tickers:
        try:
            serie = fetch_con_reintentos(obtener_historico_iol, simbolo=ticker, fecha_desde=fecha_desde, fecha_hasta=fecha_hasta, access_token=access_token)
            for periodo, dias in periodos_dias.items():
                r = rendimiento_periodo_on(serie, dias)
                if r is None:
                    registrar_error('obtener_rendimientos_ons', ticker, 'sin_historia_suficiente', f'Sin dato cercano a {periodo}')
                    continue
                resultados.append({'categoria': 'on', 'simbolo': ticker, 'periodo': periodo, 'dias': dias, 'rendimiento_periodo': r})
        except Exception as e:
            registrar_error('obtener_rendimientos_ons', ticker, 'fetch_fallido', str(e))
    return resultados

tickers_ons = ['TSC3O', 'BYCHO', 'YM34O', 'YMCXO', 'TLCPO']

# ============================================================
# EJECUCIÓN DEL PIPELINE
# ============================================================
try:
    df_inflacion = obtener_inflacion_mensual()
    inflacion_por_periodo = {p: inflacion_acumulada_ultimos_n_meses(df_inflacion, m) for p, m in periodo_a_meses.items()}

    df_plazo_fijo_raw = obtener_plazo_fijo_30_dias()
    resultados_pf = [{'categoria': 'plazo_fijo', 'simbolo': 'Depósito 30 días', 'periodo': p, 'dias': d, 'rendimiento_periodo': rendimiento_plazo_fijo(df_plazo_fijo_raw, d)} for p, d in periodos.items()]

    resultados_mm = obtener_rendimientos_fondos_cafci(fondos_fci, 'fci')
    resultados_cedears = obtener_rendimientos_ba(['KO', 'AAPL', 'MELI'], periodos, 'cedear')
    resultados_ar = obtener_rendimientos_ba(['GGAL', 'PAMP', 'YPFD'], periodos, 'accion_local_ar')

    df_usa = pd.DataFrame(obtener_rendimientos_acciones_usa(['SPY', 'AMZN', 'TSLA', 'NVDA'], periodos, 'accion_usa'))

    df_dolar = obtener_dolar_mep()
    resultados_usa_ars = obtener_rendimientos_acciones_usa_en_ars(['SPY', 'AMZN', 'TSLA', 'NVDA'], periodos, df_dolar)

    IOL_USERNAME = 'tu_usuario_iol'      # <-- reemplazar
    IOL_PASSWORD = 'tu_contraseña_iol'   # <-- reemplazar
    access_token = obtener_token_iol(IOL_USERNAME, IOL_PASSWORD)
    resultados_ons = obtener_rendimientos_ons(tickers_ons, periodos, access_token)

    df_ars = pd.DataFrame(resultados_pf + resultados_mm + resultados_cedears + resultados_ar + resultados_ons + resultados_usa_ars)
    df_ars['instrumento_dolarizado'] = df_ars['instrumento_dolarizado'].fillna(False)
    df_ars['inflacion_periodo'] = df_ars['periodo'].map(inflacion_por_periodo)
    df_ars['rendimiento_real'] = (1 + df_ars['rendimiento_periodo']) / (1 + df_ars['inflacion_periodo']) - 1
    df_ars['orden_periodo'] = df_ars['periodo'].map({'1 mes': 1, '3 meses': 2, '6 meses': 3, '12 meses': 4})
    df_ars.insert(0, 'id', range(1, len(df_ars) + 1))

    ws_consolidado.clear()
    ws_consolidado.update([df_ars.columns.values.tolist()] + df_ars.astype(str).values.tolist())
    ws_usa = sheet.worksheet('Acciones_USA_USD')
    ws_usa.clear()
    ws_usa.update([df_usa.columns.values.tolist()] + df_usa.astype(str).values.tolist())

    filas_procesadas = len(df_ars) + len(df_usa)
    ws_log.append_row([fecha_hora_corrida, 'pipeline_proyecto3', 'multiple', filas_procesadas, 'OK_CON_ERRORES' if detalle_errores_local else 'OK', f'{len(detalle_errores_local)} errores'])
    escribir_detalle_errores()

    print('Pipeline completo. Filas en df_ars:', len(df_ars))
    print(df_ars)
    print('\nAcciones USA en USD (informativo, no comparado contra inflación ARS):')
    print(df_usa)

    print(df_dolar.head(3))  # el valor 'de hoy' que usa la función
    fecha_objetivo_test = df_dolar['fecha'].max() - pd.Timedelta(days=30)
    df_dolar_test = df_dolar.copy()
    df_dolar_test['diff'] = (df_dolar_test['fecha'] - fecha_objetivo_test).abs()
    print(df_dolar_test.loc[df_dolar_test['diff'].idxmin()])  # el valor 'de hace 1 mes' que usa la función

except Exception as e:
    ws_log.append_row([fecha_hora_corrida, 'pipeline_proyecto3', 'multiple', 0, 'ERROR', str(e)])
    print('ERROR GENERAL DEL PIPELINE:')
    traceback.print_exc()